In [ ]:
### Liberias ###

# Data wraling
import pandas as pd

# API
import requests

# Base de datos
import pyodbc


import warnings
warnings.filterwarnings("ignore")

In [2]:
# URL base de la API
BASE_URL = 'https://economia.awesomeapi.com.br/json/last/USD-BRL,EUR-BRL,BTC-BRL'

# Realizar una solicitud a la API
response = requests.get(BASE_URL)

# Verificar que la solicitud fue exitosa
if response.status_code == 200:
    data = response.json()

    # Convertir los datos en un DataFrame, con claves como filas en crudo
    df_raw = pd.DataFrame.from_dict(data, orient='index')

    # Nomalizo dataframe
    df = pd.DataFrame({
        'moeda_base': df_raw['code'],
        'moeda_destino': df_raw['codein'],
        'valor_compra': df_raw['bid'].astype(float).round(2),
        'valor_venda': df_raw['ask'].astype(float).round(2),
        'data_hora': pd.to_datetime(df_raw['create_date'], utc=True).dt.strftime('%Y-%m-%d %H:%M:%S')
        })

    # Mostrar los primeros registros para verificar
    print(df)

else:
    print(f"Error en la solicitud: {response.status_code}")


       moeda_base moeda_destino  valor_compra  valor_venda  \
USDBRL        USD           BRL          5.67         5.67   
EURBRL        EUR           BRL          6.39         6.41   
BTCBRL        BTC           BRL     550712.00    550713.00   

                  data_hora  
USDBRL  2025-05-01 18:42:36  
EURBRL  2025-05-01 18:39:55  
BTCBRL  2025-05-01 18:42:24  


In [3]:
# Doy salida en un archivo csv
df.to_csv('dados_moedas.csv',index=False)

In [ ]:
### Inserto en una hipotetica base de datos SQL ###
''' a su vez valido que no existan duplicados por 
 moeda_base, moeda_venta, valor_compra, valor_venta, año, mes, dia, hora y minuto ''' 

# Conecto con base de datos
conn = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=INSERTAR_SERVIDOR;'
        'DATABASE=CotizacionMonedas;'
        'UID=USUARIO;'
        'PWD=Pass;'
    )
cursor = conn.cursor()

# Creo la tabla si no existe
cursor.execute('''
        IF OBJECT_ID('dbo.cotizaciones', 'U') IS NULL
        CREATE TABLE dbo.cotizaciones (
            moeda_base VARCHAR(16),
            moeda_destino VARCHAR(16),
            valor_compra FLOAT,
            valor_venda FLOAT,
            data_hora DATETIME,
            CONSTRAINT uq_cotizaciones UNIQUE (moeda_base, moeda_destino, valor_compra, valor_venda, data_hora)
        )
    ''')
conn.commit()


In [ ]:
# Empiezo a insertar
for _, row in df.iterrows():
    try:
        cursor.execute('''
            INSERT INTO dbo.cotizaciones (moeda_base, moeda_destino, valor_compra, valor_venda, data_hora)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            row['moeda_base'],
            row['moeda_destino'],
            row['valor_compra'],
            row['valor_venda'],
            row['data_hora']
        ))
        print(f"Procesado: {row.to_dict()}")
    except pyodbc.IntegrityError as e:
        print(f"Duplicado no insertado: {row.to_dict()} - Error: {str(e)}")

# Confirmar los cambios
conn.commit()
conn.close()